# Marketing LLM — QLoRA Fine-Tune on Kaggle

Fine-tunes **Llama 3.1 8B Instruct** on marketing-specific Q&A pairs using Unsloth (2-5× faster than vanilla transformers).

## Requirements
- **Accelerator:** GPU T4 ×2 (default on Kaggle) or P100 — set in **Settings → Accelerator**
- **Internet:** ON (required for model + dataset download) — toggle in **Settings → Internet**
- **Persistence:** Files only (default)

## Inputs
- `training/data/marketing_sft.jsonl` — 200 synthetic examples (committed to the repo)
- Optionally merged with Supabase feedback data (pulled at train-time)

## Outputs
- `marketing-llm-8b-lora/` — LoRA adapter (~150MB)
- `marketing-llm-8b-merged/` — Merged full weights (~16GB, optional)
- Pushed to Hugging Face Hub under your username

**Expected runtime:** ~90-150 min on T4×2, ~60 min on P100

## 1 · Pre-flight check + install dependencies

**This cell first verifies that a GPU is actually available, then installs Unsloth.**

If you see `AssertionError: GPU not enabled` — stop and:
1. Phone-verify your Kaggle account at https://www.kaggle.com/settings (one-time, ~30s)
2. Settings → Accelerator → **GPU T4 ×2**
3. Run → **Factory reset session** (changing the accelerator alone does NOT restart — you must reset)
4. Re-run this cell

The pip install uses Unsloth's official Kaggle channel which auto-detects the CUDA version on the host and pulls the matching `bitsandbytes` wheel. Earlier pinned versions caused `Could not find the bitsandbytes CUDA binary` errors on Kaggle's current base image.

In [ ]:
# ── Pre-flight: GPU must be available ──────────────────────────────────
import torch
assert torch.cuda.is_available(), (
    "❌ No GPU. Settings → Accelerator → GPU T4×2, then Run → Factory reset session."
)
print(f"✓ CUDA {torch.version.cuda} — {torch.cuda.device_count()} GPU(s), torch {torch.__version__}")

# ── Robust install for current Kaggle (CUDA 12.8 / PyTorch 2.6+) ───────
# Strategy: uninstall any pre-existing Unsloth bits first, then install
# the matched bundle. This avoids stale wheel cache + transformers
# version drift causing ImportErrors inside unsloth.models.llama.
import subprocess, sys

def run(*args, check=True):
    return subprocess.run(
        [sys.executable, "-m", "pip", *args],
        check=check, capture_output=True, text=True
    )

# 1. Clean slate for Unsloth + bnb to avoid mixed-version chimeras
run("uninstall", "-y", "unsloth", "unsloth_zoo", "bitsandbytes", check=False)

# 2. Install in the order Unsloth's official Kaggle guide uses
print("Installing bitsandbytes (CUDA 12.8 binaries)...")
run("install", "-q", "--upgrade", "--no-cache-dir", "bitsandbytes>=0.46.1")

print("Installing transformers (Unsloth-compatible)...")
run("install", "-q", "--upgrade", "transformers>=4.49.0,<4.55.0")

print("Installing peft / trl / accelerate...")
run("install", "-q", "--upgrade", "peft>=0.14.0", "trl>=0.12.0,<0.13.0", "accelerate>=1.2.0")

print("Installing unsloth + unsloth_zoo from main...")
run("install", "-q", "--upgrade", "--no-cache-dir",
    "git+https://github.com/unslothai/unsloth.git@main",
    "git+https://github.com/unslothai/unsloth_zoo.git@main")

print("Installing datasets + huggingface_hub...")
run("install", "-q", "--upgrade", "datasets", "huggingface_hub")

# 3. Verify bnb actually imports with CUDA support
import sys
for m in list(sys.modules):
    if m.startswith(("bitsandbytes", "unsloth", "transformers", "peft", "trl")):
        sys.modules.pop(m, None)

import bitsandbytes as bnb
print(f"✓ bitsandbytes {bnb.__version__} loaded")

print("\n" + "=" * 60)
print("⚠ IMPORTANT: Click Run → Restart Session NOW before running")
print("  the next cell. Python needs a clean import after this many")
print("  package upgrades.")
print("=" * 60)

## 2 · Load base model (Llama 3.1 8B Instruct) with 4-bit quantization

4-bit quantization brings VRAM use from ~16GB → ~5GB so it fits on a single T4 with headroom for the LoRA adapters.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"Model loaded. GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 3 · Attach LoRA adapters

We train only the adapter layers (~0.5% of total params). This is what gets saved + reused.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # LoRA rank — 16 is the sweet spot
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4 · Load + format training data

Pull the JSONL dataset. If you uploaded it as a Kaggle dataset, change the path. If using the GitHub repo, run the git clone cell first.

In [ ]:
# Option A: Clone from your GitHub repo (recommended)
import os
if not os.path.exists('/kaggle/working/Marketing'):
    !git clone -q https://github.com/amittomar-hue/Marketing.git /kaggle/working/Marketing

DATA_PATH = '/kaggle/working/Marketing/training/data/marketing_sft.jsonl'
print(f"Dataset: {DATA_PATH}")
!wc -l $DATA_PATH

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"Examples: {len(raw)}")
print(f"Sample: {raw[0]}")

# Format each example into Llama 3.1 chat template
def format_example(ex):
    messages = [
        {"role": "system", "content": "You are Marketing LLM, an enterprise-grade marketing assistant. Be direct, specific, and data-driven. Format responses in clean markdown."},
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = raw.map(format_example, remove_columns=raw.column_names)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(ds['train'])}, Eval: {len(ds['test'])}")

## 5 · Train

Hyperparameters tuned for ~200-500 example datasets on T4/P100. Larger datasets: increase `num_train_epochs` to 1 and `max_steps` to -1.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["test"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,    # Effective batch = 8
        warmup_steps = 5,
        num_train_epochs = 3,                # 3 epochs over ~200 examples = ~75 steps
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "/kaggle/working/checkpoints",
        report_to = "none",
    ),
)

stats = trainer.train()
print(f"\nTraining complete. Loss: {stats.training_loss:.4f}")

## 6 · Test the fine-tuned model

Quick sanity check before saving.

In [ ]:
# Free up VRAM left over from training — otherwise inference will OOM on T4
import gc, torch
gc.collect()
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"Free VRAM after cleanup: {free_gb:.2f} GB")

FastLanguageModel.for_inference(model)

def generate(prompt, max_new_tokens=300):
    messages = [
        {"role": "system", "content": "You are Marketing LLM, an enterprise-grade marketing assistant."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.inference_mode():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print(generate("Write 3 Google Ads variants for an AI marketing platform targeting growth marketers at D2C brands."))
print("\n" + "=" * 60 + "\n")
print(generate("What are 3 emerging trends in B2B marketing for next quarter?"))

## 7 · Save the LoRA adapter locally

The adapter is ~150MB. The merged full model is ~16GB.

In [ ]:
ADAPTER_DIR = "/kaggle/working/marketing-llm-8b-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")
!du -sh $ADAPTER_DIR

## 8 · Push to Hugging Face Hub (optional but recommended)

Makes the model loadable from anywhere via `model_name="<your-username>/marketing-llm-8b-lora"`.

1. Get your HF token: https://huggingface.co/settings/tokens (Create new token, role=Write)
2. Add it to Kaggle Secrets: **Add-ons → Secrets → Add Secret** with key `HF_TOKEN`

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    HF_USERNAME = "amittomar-hue"  # ← change to your HF username if different
    REPO_NAME = f"{HF_USERNAME}/marketing-llm-8b-lora"
    model.push_to_hub(REPO_NAME, token=hf_token)
    tokenizer.push_to_hub(REPO_NAME, token=hf_token)
    print(f"\n✓ Pushed to https://huggingface.co/{REPO_NAME}")
except Exception as e:
    print(f"Skipping HF push: {e}")
    print("Adapter still saved locally at /kaggle/working/marketing-llm-8b-lora — download via Kaggle UI")

## 9 · (Optional) Push merged + GGUF for inference

Merged weights + GGUF quantizations let you run the model anywhere — Ollama, llama.cpp, vLLM, Together.ai.

Only run this if you plan to serve the model outside of HF Inference API.

In [ ]:
# Merged 16-bit weights — for vLLM, Together.ai serving
# model.save_pretrained_merged("/kaggle/working/marketing-llm-8b-merged", tokenizer, save_method="merged_16bit")

# GGUF Q4_K_M — for Ollama / llama.cpp (smallest, fastest)
# model.save_pretrained_gguf("/kaggle/working/marketing-llm-8b-gguf", tokenizer, quantization_method="q4_k_m")

print("Uncomment the cells above to produce merged + GGUF artifacts.")

## Next steps

1. **Download the adapter** from Kaggle output OR pull from HF Hub
2. **Wire up inference** in your Marketing LLM frontend — use HF Inference API, Replicate, or self-host
3. **Collect more feedback** via the Supabase RLMO loop, then re-train monthly with the merged dataset

See `/training/README.md` in the repo for the full pipeline including how to merge new Supabase feedback into your training data.